Every DCF starts with a revenue forecast, and most start from the company's own history: recent growth, fading toward the economy. This notebook asks how accurate that is, and whether the range around it is honest.

Each number below is computed in the cell above it from the published results file. The pipeline that builds that file from SEC filings is in [`scripts/run_forecast_accuracy.py`](https://github.com/Courtizy/research/blob/main/scripts/run_forecast_accuracy.py).

In [1]:
#| label: setup
#| code-summary: "Setup: load the results and the brand chart style"
import json, math, sys
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, HTML, display

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "brand"))
import palette                                     # Decision Models colors, validated for color-blind safety

pio.templates["dm"] = palette.plotly_template("valuation", "light")
pio.templates.default = "simple_white+dm"
COLORS = dict(zip(["model", "hist3", "sector", "naive"], palette.series("valuation", "light", 4)))
NAMES = {"model": "Model (fade)", "hist3": "Own 3-yr growth", "sector": "Sector growth", "naive": "No change"}

R = json.loads((ROOT / "site" / "data" / "forecast_accuracy.json").read_text())
acc = pd.DataFrame(R["accuracy"])
cov = pd.DataFrame(R["coverage"]).set_index("h")
pct = lambda x: f"{x:.0%}"

## The answer

In [2]:
#| echo: false
display(Markdown(f"**{R['headline']['lead']}** {R['headline']['sub']}"))
display(HTML('<div class="kpis">' + "".join(f'<div class="kpi"><b>{k["value"]}</b><span>{k["label"]}</span></div>'
                                            for k in R["headline"]["kpis"]) + "</div>"))

**Three years out, a forecast built from a company's own filings misses actual revenue by a median 20%, and the model's 80% range contains the outcome 14% of the time.** The model's fading-growth forecast is about as accurate as assuming no change (1% more error, t = 0.6, not a real difference). Extrapolating a company's own three-year growth does clearly worse: 23% more error (t = 6.6). The range around the model is far narrower than the uncertainty it describes.

## Data and design

- **Universe:** the 300 largest US filers by revenue (at least $1B) in 2011, 2015 and 2019, taken together, so companies that were later acquired or stopped filing stay in. Finance (SIC 60–67) is excluded.
- **Point in time, both ways:** each forecast is made on the day a 10-K was filed, using only facts filed by then. Each actual is revenue as first reported, so later restatements never leak in.
- **Four forecasts** of revenue 1–5 years out: no change, the company's own three-year growth, its sector's median growth, and the Valuation Model's fading-growth path with its P10–P90 range.

In [3]:
s = R["sample"]
pd.Series({"Companies in the universe": s["universe"], "Excluded (finance)": s["excluded_finance"],
           "Companies scored": s["companies_scored"], "Forecast origins": s["origins"],
           "Forecast–outcome pairs": s["observations"],
           "Base years": f'{s["base_years"][0]}–{s["base_years"][1]}'}, name="").to_frame()

,
Companies in the universe,437
Excluded (finance),60
Companies scored,359
Forecast origins,4514
Forecast–outcome pairs,16618
Base years,2009–2025


## Accuracy

Error is the log ratio of forecast to actual (0.10 is about 10% off). Skill compares mean absolute error with the no-change forecast on the same observations; the *t* statistic clusters by company.

In [4]:
#| label: fig-error
#| fig-cap: "Median absolute error by horizon. The model tracks no change; extrapolating own growth is worst."
fig = go.Figure()
for m in ["model", "hist3", "sector", "naive"]:
    d = acc[acc.method == m]
    fig.add_scatter(x=d.h, y=d.median_abs_error, name=NAMES[m], mode="lines+markers", line_color=COLORS[m])
fig.update_layout(xaxis_title="Years out", yaxis_title="Median absolute log error", height=380,
                  legend=dict(orientation="h", y=-0.25), margin=dict(t=10))
fig.show()

In [5]:
t = acc[acc.h.isin([1, 3, 5])].assign(
    Method=lambda d: d.method.map(NAMES),
    Skill=lambda d: d.skill_vs_naive.map(pct),
    t=lambda d: d.loss_diff_vs_naive.map(lambda x: round(x["t"], 1) if x and x.get("t") is not None else None))
t.pivot(index="Method", columns="h", values="Skill").rename(columns=lambda h: f"{h} yr skill")

h,1 yr skill,3 yr skill,5 yr skill
Method,,,
Model (fade),0%,-1%,2%
No change,0%,0%,0%
Own 3-yr growth,-15%,-23%,-29%
Sector growth,-1%,1%,-1%


## Is the range honest?

In [6]:
#| label: fig-coverage
#| fig-cap: "Share of outcomes inside the model's P10–P90 range. It promises 80%."
fig = go.Figure(go.Bar(x=cov.index, y=cov.coverage, marker_color=COLORS["model"], name="Inside the range"))
fig.add_hline(y=0.8, line_dash="dash", annotation_text="promised 80%")
fig.update_layout(xaxis_title="Years out", yaxis_tickformat=".0%", yaxis_range=[0, 1], height=340, margin=dict(t=10))
fig.show()

## Who gets over-forecast?

In [7]:
#| label: fig-bias
#| fig-cap: "Mean signed error three years out by prior-growth decile. Above zero = forecast too high."
b = pd.DataFrame(R["bias_by_growth"])
fig = go.Figure()
for m in ["model", "hist3"]:
    fig.add_scatter(x=b.group, y=b[m].map(lambda x: x["mean"]), name=NAMES[m], mode="lines+markers", line_color=COLORS[m],
                    error_y=dict(array=b[m].map(lambda x: 2 * x["se"]), thickness=1))
fig.add_hline(y=0, line_color="#999")
fig.update_layout(xaxis_title="Prior growth decile (1 = slowest)", yaxis_title="Mean log error", height=360,
                  legend=dict(orientation="h", y=-0.25), margin=dict(t=10))
fig.show()

## Hypotheses, set before looking

In [8]:
#| echo: false
display(Markdown("\n".join(f"- **{f['label']}.** {f['text']}" for f in R["headline"]["findings"])))

- **H1 supported.** For the fastest growers (top tenth, median prior growth 27%), the model's three-year forecast averaged 23% too high (t = 3.7); for the slowest tenth, 27% too low.
- **H2 supported.** Three years out, holding last year's operating margin missed by a median 2.8 points against 2.9 for the three-year average (t = -1.3, negative favors the average).
- **H3 partly supported.** The P10–P90 range held the outcome 14% of the time one year out and 14% five years out, against the 80% it implies. Misses fell 45% below the range and 41% above it three years out.

## What it can't tell you

- It tests mechanical forecasts built only from filings. Analysts and management know more.
- Associations, not causes.
- Large US filers only; small and private companies may behave differently.

## Reproduce

```bash
pip install -e ".[dev]" && scripts/setup_engine.sh
export SEC_USER_AGENT="Your Name you@example.com"
python scripts/run_forecast_accuracy.py
```